# Sesi 1 · LLM tanpa pustaka, lalu RAG dengan tangan

Workshop Navasena × UGM

Di notebook ini kita melihat dua hal:
1. Apa yang terjadi ketika LLM ditanya sesuatu yang tidak ada di data latihnya.
2. Ide dasar RAG, dikerjakan dengan tangan: tempelkan potongan dokumen ke prompt, baru minta jawaban.

Jalankan sel dari atas ke bawah dengan **Shift + Enter**.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from rag import db, llm, pipeline, safety
conn = db.connect()
print("Pustaka:", db.stats(conn), "| key LLM terisi:", llm.has_key())

## 1. Memanggil LLM

Kita memakai OpenRouter dengan model `openrouter/free`. Router ini memilih salah satu model gratis
untuk setiap panggilan, jadi model yang menjawab bisa berbeda dari waktu ke waktu.

In [ ]:
jawaban = llm.chat([{"role": "user", "content": "Jelaskan machine learning dalam dua kalimat."}])
print("Dijawab oleh:", jawaban.model)
print(jawaban.text)

## 2. Pertanyaan yang tidak mungkin diketahui model

Pustaka kita memuat satu dokumen internal fiktif tentang sebuah lab. Isinya tidak pernah ada di internet,
jadi model tidak mungkin mempelajarinya. Perhatikan cara model menjawab: mengaku tidak tahu, atau mengarang?

In [ ]:
pertanyaan = "Berapa anggaran komputasi Lab RAG semester ini, dan berapa yang dialokasikan untuk sewa GPU?"

tanpa_rag = pipeline.ask(conn, pertanyaan, use_rag=False, log=False)
print("Dijawab oleh:", tanpa_rag.model, "\n")
print(tanpa_rag.text)

## 3. RAG dengan tangan

Sekarang kita ambil sendiri paragraf yang relevan dari dokumen, lalu menempelkannya ke prompt.
Tiga langkah ini adalah inti RAG:

| Langkah | Yang kita lakukan di sini |
|---|---|
| **Retrieve** | membaca berkas dan mengambil bagian "Anggaran komputasi" |
| **Augment** | menempelkan bagian itu ke prompt |
| **Generate** | meminta LLM menjawab berdasarkan teks tersebut |

In [ ]:
dokumen = (ROOT / "data" / "extra" / "catatan_internal_lab.md").read_text()
awal = dokumen.index("## Anggaran komputasi")
konteks = dokumen[awal : dokumen.index("## Hasil uji internal")]
print(konteks)

In [ ]:
prompt = f"""Jawab pertanyaan hanya berdasarkan konteks berikut.

<konteks>
{konteks}
</konteks>

Pertanyaan: {pertanyaan}"""

jawaban = llm.chat([{"role": "user", "content": prompt}])
print("Dijawab oleh:", jawaban.model, "\n")
print(jawaban.text)

## 4. Yang belum otomatis

Jawabannya sekarang benar, tetapi kita sendiri yang mencari paragrafnya. Untuk ratusan dokumen, pencarian itu
harus dikerjakan mesin. Itulah yang dibangun di Sesi 2: dokumen dipotong, diubah menjadi vektor, lalu disimpan
di basis data yang bisa mencari berdasarkan kemiripan makna.

### Coba sendiri
- Ganti `pertanyaan` dengan hal lain yang hanya ada di dokumen internal (jadwal GPU, hasil uji).
- Hapus kalimat "hanya berdasarkan konteks berikut" dari prompt. Apakah jawabannya berubah?
- Jalankan sel nomor 2 beberapa kali. Apakah model yang menjawab selalu sama?